# Teil 3: Modellierung - Airbnb NYC 2019

In diesem Notizbuch teile ich die Daten auf, trainiere ein Modell und prüfe die Vorhersagen auf Plausibilität.

## 3.1 Train- und Test-Satz

Ich verwende `price` als Zielvariable und trenne die Daten in Trainings- und Testdaten.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('AB_NYC_2019.csv')

# Unplausible Preise (<= 0) entfernen, da sie die Modellgüte verfälschen.
anzahl_vorher = len(df)
df = df[df['price'] > 0].copy()
anzahl_entfernt = anzahl_vorher - len(df)

feature_spalten = [
    'neighbourhood_group',
    'room_type',
    'latitude',
    'longitude',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365'
]
target_spalte = 'price'

daten = df[feature_spalten + [target_spalte]].copy()
X = daten[feature_spalten]
y = daten[target_spalte]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f'Trainingsdaten: {X_train.shape[0]} Zeilen')
print(f'Testdaten: {X_test.shape[0]} Zeilen')
print(f'Entfernte Zeilen mit price <= 0: {anzahl_entfernt}')

Trainingsdaten: 39107 Zeilen
Testdaten: 9777 Zeilen
Entfernte Zeilen mit price <= 0: 11


## 3.2 Modellwahl

Ich verwende einen RandomForestRegressor aus sklearn. Er ist für dieses Problem geeignet, weil sowohl numerische als auch kategoriale Felder vorhanden sind und der Zusammenhang zwischen Lage, Raumtyp und Preis nicht linear ist. Der Random Forest kann solche Muster erfassen, ist robust gegen Ausreisser und benötigt keine Skalierung der Eingabedaten. Mit einer Pipeline aus Imputation und One-Hot-Encoding bleibt die Vorverarbeitung sauber und reproduzierbar.

In [2]:
numerische_spalten = [
    'latitude',
    'longitude',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365'
]
kategorische_spalten = ['neighbourhood_group', 'room_type']

numerische_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

kategorische_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

vorverarbeitung = ColumnTransformer(
    transformers=[
        ('num', numerische_pipeline, numerische_spalten),
        ('cat', kategorische_pipeline, kategorische_spalten)
    ]
)

modell = Pipeline(steps=[
    ('vorverarbeitung', vorverarbeitung),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

modell.fit(X_train, y_train)

y_vorhersage = modell.predict(X_test)
test_mae = mean_absolute_error(y_test, y_vorhersage)
test_rmse = mean_squared_error(y_test, y_vorhersage) ** 0.5
test_r2 = r2_score(y_test, y_vorhersage)

test_resultate = pd.DataFrame({
    'tatsächlich': y_test.reset_index(drop=True),
    'vorhersage': y_vorhersage
})

print('Test-Kennzahlen:')
print(f'MAE:  {test_mae:.2f}')
print(f'RMSE: {test_rmse:.2f}')
print(f'R2:   {test_r2:.3f}')

Test-Kennzahlen:
MAE:  65.58
RMSE: 231.78
R2:   0.112


## 3.3 Vorhersagen prüfen

Ich prüfe die Vorhersagen manuell anhand konkreter Einzelfälle und bewerte sie auf Sinnhaftigkeit im realen Kontext. Dazu vergleiche ich tatsächlichen Preis, Prognose, Stadtteil und Unterkunftstyp für günstige, mittlere und sehr teure Angebote. So wird sichtbar, wo das Modell plausibel schätzt und wo es systematisch danebenliegt.

In [3]:
test_prüfung = X_test.reset_index(drop=True).copy()
test_prüfung['tatsächlich'] = y_test.reset_index(drop=True)
test_prüfung['vorhersage'] = y_vorhersage
test_prüfung['abweichung'] = test_prüfung['vorhersage'] - test_prüfung['tatsächlich']

# Drei konkrete Einzelfälle für die manuelle Sinnhaftigkeitsprüfung
fall_günstig = test_prüfung.iloc[(test_prüfung['tatsächlich'] - 65).abs().argsort()[:1]]
fall_mittel = test_prüfung.iloc[(test_prüfung['tatsächlich'] - 150).abs().argsort()[:1]]
fall_teuer = test_prüfung.nlargest(1, 'tatsächlich')

einzelfälle = pd.concat([fall_günstig, fall_mittel, fall_teuer]).drop_duplicates().copy()

print('Einzelfälle für die manuelle Beurteilung:')
print(
    einzelfälle[
        ['neighbourhood_group', 'room_type', 'tatsächlich', 'vorhersage', 'abweichung']
    ].to_string(index=False)
)

Einzelfälle für die manuelle Beurteilung:
neighbourhood_group       room_type  tatsächlich  vorhersage  abweichung
           Brooklyn    Private room           65      78.070      13.070
             Queens Entire home/apt          150     129.965     -20.035
          Manhattan Entire home/apt        10000     305.385   -9694.615


Manuelle Beurteilung: Der Fall Brooklyn, Private room mit 65 USD und Prognose 78.07 USD ist plausibel, weil die Grössenordnung für ein Privatzimmer passt. Der Fall Queens, Entire home/apt mit 150 USD und Prognose 129.97 USD ist ebenfalls nachvollziehbar; die Abweichung von rund 20 USD ist moderat. Beim Fall Manhattan, Entire home/apt mit 10’000 USD und Prognose 305.39 USD ist die Schätzung klar unplausibel. Fazit: Für typische Preise funktioniert das Modell brauchbar, extreme Luxus-Ausreisser unterschätzt es jedoch stark.